In [4]:
!pip install pytesseract qiskit qiskit-aer qiskit-algorithms fastapi uvicorn nest_asyncio networkx scikit-learn pandas matplotlib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.0 MB/s eta 0:00:00


In [5]:
# Foundations
import hashlib, json, uuid, math, re, io, base64, warnings, os, random, string
from datetime import datetime, timezone, timedelta
from enum import Enum
from typing import Optional, List, Dict, Any, Tuple
from collections import defaultdict, OrderedDict, deque
from dataclasses import dataclass, field, asdict
import heapq, itertools, functools

# Data & ML
import numpy as np
import pandas as pd
import networkx as nx
import cv2, pytesseract
from PIL import Image
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Quantum
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Sampler
from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit_algorithms.state_fidelities import ComputeUncompute

# Web
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import nest_asyncio

# Plots
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
nest_asyncio.apply()

In [6]:
class NISTPhase(str, Enum):
    PREPARATION    = "1. Preparation"
    DETECTION      = "2. Detection & Analysis"
    CONTAINMENT    = "3. Containment, Eradication & Recovery"
    POST           = "4. Post-Incident Activity"

# --- Curated playbook: every step mapped to a NIST phase + ATT&CK technique ---
PLAYBOOK = {
    "ransomware": {
        "description": "Ransomware detected on endpoint / file-share.",
        "severity": "critical",
        "regulatory_trigger": True,   # personal data likely impacted
        "steps": [
            # Phase 1 — Preparation
            {"phase": NISTPhase.PREPARATION, "action": "Activate IR team and bridge-line",
             "attack": "N/A", "evidence": "War-room log entry"},
            {"phase": NISTPhase.PREPARATION, "action": "Snapshot volatile memory of affected host",
             "attack": "T1486", "evidence": "Memory image + SHA256"},

            # Phase 2 — Detection & Analysis
            {"phase": NISTPhase.DETECTION, "action": "Collect ransomware note + file hashes",
             "attack": "T1486", "evidence": "Ransom note file + hashes"},
            {"phase": NISTPhase.DETECTION, "action": "Identify patient-zero host and time of encryption",
             "attack": "T1486", "evidence": "EDR timeline export"},
            {"phase": NISTPhase.DETECTION, "action": "Search for lateral-movement artefacts (SMB, RDP, WMI)",
             "attack": "T1021.002", "evidence": "Netflow + EDR logs"},

            # Phase 3 — Containment, Eradication & Recovery
            {"phase": NISTPhase.CONTAINMENT, "action": "Network-isolate infected hosts",
             "attack": "T1486", "evidence": "Firewall ACL change ticket"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Disable compromised accounts and force password reset",
             "attack": "T1078", "evidence": "AD change log"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Restore from last known-good backup (verify integrity)",
             "attack": "T1490", "evidence": "Restore report + hash comparison"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Patch the exploited vector",
             "attack": "T1190", "evidence": "Patch management ticket"},

            # Phase 4 — Post-Incident
            {"phase": NISTPhase.POST, "action": "Conduct lessons-learned review",
             "attack": "N/A", "evidence": "Post-incident report"},
            {"phase": NISTPhase.POST, "action": "Update playbooks and detection rules",
             "attack": "N/A", "evidence": "Confluence / Sigma rule commit"},
        ]
    },
    "phishing": {
        "description": "Credential-phishing campaign targeting employees.",
        "severity": "high",
        "regulatory_trigger": True,
        "steps": [
            {"phase": NISTPhase.PREPARATION, "action": "Notify IR lead and mail admin",
             "attack": "N/A", "evidence": "Ticket"},
            {"phase": NISTPhase.DETECTION, "action": "Pull phishing email headers and URL",
             "attack": "T1566.002", "evidence": "Email .eml file"},
            {"phase": NISTPhase.DETECTION, "action": "Identify all recipients and clickers",
             "attack": "T1566.002", "evidence": "Mail-gateway report"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Block sender domain and URL at gateway",
             "attack": "T1566.002", "evidence": "Gateway policy change"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Force-reset passwords of clickers, revoke sessions",
             "attack": "T1078", "evidence": "IdP audit log"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Enable MFA on affected accounts",
             "attack": "T1556", "evidence": "MFA enrollment log"},
            {"phase": NISTPhase.POST, "action": "Run awareness training for clickers",
             "attack": "N/A", "evidence": "LMS completion report"},
            {"phase": NISTPhase.POST, "action": "Tune mail filters / DMARC",
             "attack": "N/A", "evidence": "Mail policy diff"},
        ]
    },
    "data_exfiltration": {
        "description": "Suspected large outbound data transfer to untrusted host.",
        "severity": "critical",
        "regulatory_trigger": True,
        "steps": [
            {"phase": NISTPhase.PREPARATION, "action": "Engage legal + DPO immediately",
             "attack": "N/A", "evidence": "Legal memo"},
            {"phase": NISTPhase.DETECTION, "action": "Confirm exfil destination and volume",
             "attack": "T1041", "evidence": "Netflow / firewall log"},
            {"phase": NISTPhase.DETECTION, "action": "Identify what data was accessed (DLP + DB audit)",
             "attack": "T1039", "evidence": "DLP incident export"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Block destination IPs / domains",
             "attack": "T1041", "evidence": "Firewall change"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Revoke compromised credentials and tokens",
             "attack": "T1528", "evidence": "IdP log"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Preserve forensic image of affected servers",
             "attack": "N/A", "evidence": "Forensic image + hash"},
            {"phase": NISTPhase.POST, "action": "File GDPR Article 33 notification if personal data confirmed",
             "attack": "N/A", "evidence": "Regulator receipt"},
            {"phase": NISTPhase.POST, "action": "Update DLP rules and egress monitoring",
             "attack": "N/A", "evidence": "DLP policy diff"},
        ]
    },
    "insider_threat": {
        "description": "Employee suspected of exfiltrating IP / customer data.",
        "severity": "high",
        "regulatory_trigger": True,
        "steps": [
            {"phase": NISTPhase.PREPARATION, "action": "Loop in HR + Legal (chain-of-custody critical)",
             "attack": "N/A", "evidence": "Case file"},
            {"phase": NISTPhase.DETECTION, "action": "Review badge, VPN and DLP logs",
             "attack": "T1078", "evidence": "Log export bundle"},
            {"phase": NISTPhase.DETECTION, "action": "Preserve laptop image before any wipe",
             "attack": "N/A", "evidence": "Disk image + hash"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Suspend account (not delete) — preserve mailbox",
             "attack": "T1531", "evidence": "AD suspension log"},
            {"phase": NISTPhase.CONTAINMENT, "action": "Block USB / cloud-upload for the user group",
             "attack": "T1052.001", "evidence": "GPO change"},
            {"phase": NISTPhase.POST, "action": "Notify regulator / customer if personal data involved",
             "attack": "N/A", "evidence": "Notification draft"},
            {"phase": NISTPhase.POST, "action": "Publish lessons-learned and tighten RBAC",
             "attack": "N/A", "evidence": "RBAC change request"},
        ]
    },
}

def pick_playbook(description: str) -> Tuple[str, Dict]:
    """Naive keyword-based playbook selector."""
    d = description.lower()
    if any(k in d for k in ["ransom", "encrypt", "locked", "crypto"]):
        return "ransomware", PLAYBOOK["ransomware"]
    if any(k in d for k in ["phish", "email", "credential", "click"]):
        return "phishing", PLAYBOOK["phishing"]
    if any(k in d for k in ["exfil", "outbound", "leak", "transfer", "download"]):
        return "data_exfiltration", PLAYBOOK["data_exfiltration"]
    if any(k in d for k in ["insider", "employee", "usb", "internal"]):
        return "insider_threat", PLAYBOOK["insider_threat"]
    return "ransomware", PLAYBOOK["ransomware"]  # default

In [7]:
class EvidenceTimeline:
    """
    Chain-of-custody timeline. Every event is hash-chained (SHA3-512)
    with a quantum-derived nonce and a simulated ML-DSA signature.
    This reuses the original QuantumHashChain logic but the semantic
    is now a forensic evidence ledger.
    """

    def __init__(self, incident_id: str):
        self.incident_id = incident_id
        self.simulator = AerSimulator()
        self.chain: List[Dict] = []
        self.custody_log: List[Dict] = []   # who touched what, when

    def _quantum_nonce(self) -> str:
        qc = QuantumCircuit(8, 8)
        qc.h(range(8)); qc.measure(range(8), range(8))
        counts = self.simulator.run(qc, shots=1).result().get_counts()
        return hex(int(list(counts.keys())[0], 2))[2:].zfill(2)

    def add_event(self, actor: str, action: str, details: str,
                  nist_phase: NISTPhase, attack_id: str = "N/A",
                  artifact_hash: Optional[str] = None) -> Dict:
        prev = self.chain[-1]["event_hash"] if self.chain else "GENESIS"
        nonce = self._quantum_nonce()
        ts = datetime.now(timezone.utc).isoformat()

        payload = json.dumps({
            "incident_id": self.incident_id, "timestamp": ts,
            "actor": actor, "action": action, "details": details,
            "nist_phase": nist_phase.value, "attack_id": attack_id,
            "artifact_hash": artifact_hash,
            "previous_hash": prev, "nonce": nonce
        }, sort_keys=True)

        sha3 = hashlib.sha3_512(payload.encode()).hexdigest()
        pqc_sig = hashlib.sha3_256(f"ML-DSA:{sha3}:{nonce}".encode()).hexdigest()[:64]

        sealed = {
            "seq": len(self.chain) + 1,
            "timestamp_utc": ts,
            "actor": actor, "action": action, "details": details,
            "nist_phase": nist_phase.value,
            "attack_id": attack_id,
            "artifact_hash": artifact_hash,
            "event_hash": sha3,
            "previous_hash": prev,
            "quantum_nonce": nonce,
            "pqc_signature": pqc_sig,
            "algorithm": "SHA3-512 + ML-DSA (simulated FIPS 204)",
        }
        self.chain.append(sealed)

        # chain-of-custody entry
        self.custody_log.append({
            "seq": sealed["seq"], "timestamp_utc": ts,
            "actor": actor, "artifact_hash": artifact_hash,
            "transfer": action, "witnessed_by": "system",
        })
        return sealed

    def verify(self) -> Dict:
        ok = all(
            self.chain[i]["previous_hash"] ==
            (self.chain[i-1]["event_hash"] if i > 0 else "GENESIS")
            for i in range(len(self.chain))
        )
        return {
            "incident_id": self.incident_id,
            "total_events": len(self.chain),
            "chain_intact": ok,
            "custody_entries": len(self.custody_log),
            "quantum_safe": True,
        }

    def timeline_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.chain)

In [8]:
class RegulationEngine:
    """
    Determines which regulators must be notified and when.
    GDPR Art. 33 → 72 hours to the supervisory authority.
    GDPR Art. 34 → without undue delay to data subjects.
    """

    RULES = [
        {"name": "GDPR Art. 33 (Supervisory Authority)",
         "trigger_data_types": ["personal_data","pii","customer_data"],
         "deadline_hours": 72,
         "authority": "Lead Supervisory Authority"},
        {"name": "GDPR Art. 34 (Data Subjects)",
         "trigger_data_types": ["personal_data","pii","high_risk"],
         "deadline_hours": None,  # 'without undue delay'
         "authority": "Affected data subjects"},
        {"name": "PCI-DSS 10.7 / 12.10 (Cardholder data)",
         "trigger_data_types": ["cardholder","pan","cvv"],
         "deadline_hours": None,
         "authority": "Acquiring bank + card brands"},
        {"name": "SEC Material-Cyber (Item 1.05 8-K)",
         "trigger_data_types": ["material_incident"],
         "deadline_hours": 96,
         "authority": "SEC"},
        {"name": "NIS2 (Essential services)",
         "trigger_data_types": ["essential_services"],
         "deadline_hours": 24,
         "authority": "National CSIRT"},
    ]

    def __init__(self, incident_start_utc: datetime):
        self.start = incident_start_utc

    def evaluate(self, data_types: List[str], severity: str) -> List[Dict]:
        now = datetime.now(timezone.utc)
        elapsed_h = (now - self.start).total_seconds() / 3600.0
        hits = []
        for rule in self.RULES:
            if any(d in data_types for d in rule["trigger_data_types"]):
                if rule["deadline_hours"]:
                    remaining = max(0.0, rule["deadline_hours"] - elapsed_h)
                    status = "OVERDUE" if remaining == 0 else "PENDING"
                    due = (self.start + timedelta(hours=rule["deadline_hours"])).isoformat()
                else:
                    remaining, status, due = None, "ASAP", None
                hits.append({
                    "regulation": rule["name"],
                    "notify": rule["authority"],
                    "deadline_hours": rule["deadline_hours"],
                    "elapsed_hours": round(elapsed_h, 2),
                    "hours_remaining": None if remaining is None else round(remaining, 2),
                    "due_utc": due,
                    "status": status,
                    "severity_at_trigger": severity,
                })
        return hits

In [9]:
class AttackGraphEngine:
    """Full graph-algorithm suite (unchanged algorithms, incident-response framing)."""

    def __init__(self, G: nx.DiGraph):
        self.G = G

    def bfs(self, start):
        visited, dist, order = {start}, {start: 0}, [start]
        q = deque([start])
        while q:
            u = q.popleft()
            for v in self.G.successors(u):
                if v not in visited:
                    visited.add(v); dist[v] = dist[u] + 1
                    order.append(v); q.append(v)
        return {"algorithm": "BFS", "reachable": len(order),
                "nodes": order[:10], "complexity": "O(V + E)"}

    def dijkstra(self, src):
        dist, prev = {src: 0}, {}
        pq, visited = [(0, src)], set()
        while pq:
            d, u = heapq.heappop(pq)
            if u in visited: continue
            visited.add(u)
            for v in self.G.successors(u):
                w = self.G[u][v].get('duration', 1.0)
                nd = d + w
                if nd < dist.get(v, math.inf):
                    dist[v] = nd; prev[v] = u
                    heapq.heappush(pq, (nd, v))
        top = sorted(dist.items(), key=lambda x: x[1])[:10]
        return {"algorithm": "Dijkstra", "top_10": dict(top),
                "complexity": "O((V+E) log V)"}

    def astar(self, src, dst):
        h = lambda n: 0 if self.G.nodes[n].get('attack_label') == 1 else 100
        open_set = [(h(src), 0, src)]
        came, gs, visited = {}, {src: 0}, set()
        while open_set:
            _, g, cur = heapq.heappop(open_set)
            if cur == dst:
                path = [cur]
                while cur in came:
                    cur = came[cur]; path.append(cur)
                return {"algorithm": "A*", "path": path[::-1],
                        "cost": g, "complexity": "O(E log V)"}
            if cur in visited: continue
            visited.add(cur)
            for v in self.G.successors(cur):
                w = self.G[cur][v].get('duration', 1.0)
                t = gs[cur] + w
                if t < gs.get(v, math.inf):
                    came[v] = cur; gs[v] = t
                    heapq.heappush(open_set, (t + h(v), t, cur))
        return {"algorithm": "A*", "path": None, "error": "no path"}

    def kahn_topo(self):
        indeg = {n: self.G.in_degree(n) for n in self.G}
        q = deque([n for n, d in indeg.items() if d == 0])
        order = []
        while q:
            u = q.popleft(); order.append(u)
            for v in self.G.successors(u):
                indeg[v] -= 1
                if indeg[v] == 0: q.append(v)
        return {"algorithm": "Kahn Topological Sort", "order": order[:10],
                "is_dag": len(order) == len(self.G), "complexity": "O(V + E)"}

    def tarjan_scc(self):
        idx, stack, on = [0], [], set()
        indices, low, sccs = {}, {}, []
        def strong(v):
            indices[v] = low[v] = idx[0]; idx[0] += 1
            stack.append(v); on.add(v)
            for w in self.G.successors(v):
                if w not in indices:
                    strong(w); low[v] = min(low[v], low[w])
                elif w in on:
                    low[v] = min(low[v], indices[w])
            if low[v] == indices[v]:
                comp = []
                while True:
                    w = stack.pop(); on.remove(w); comp.append(w)
                    if w == v: break
                sccs.append(comp)
        for n in self.G:
            if n not in indices: strong(n)
        cyclic = [s for s in sccs if len(s) > 1]
        return {"algorithm": "Tarjan SCC", "total_sccs": len(sccs),
                "cyclic_sccs": len(cyclic),
                "all_sizes": [len(s) for s in sccs],
                "complexity": "O(V + E)"}

    def kruskal_mst(self):
        G_u = self.G.to_undirected()
        parent = {n: n for n in G_u}; rank = {n: 0 for n in G_u}
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]; x = parent[x]
            return x
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra == rb: return False
            if rank[ra] < rank[rb]: ra, rb = rb, ra
            parent[rb] = ra
            if rank[ra] == rank[rb]: rank[ra] += 1
            return True
        edges = sorted(G_u.edges(data=True), key=lambda e: e[2].get('duration', 1.0))
        mst = []
        for u, v, d in edges:
            if union(u, v): mst.append((u, v))
        return {"algorithm": "Kruskal MST", "mst_edges": len(mst),
                "complexity": "O(E log E)"}

    def pagerank(self, d=0.85, iters=30):
        nodes = list(self.G.nodes); n = len(nodes)
        if n == 0: return {"algorithm": "PageRank", "top": {}}
        pr = {x: 1/n for x in nodes}
        for _ in range(iters):
            new = {x: (1-d)/n for x in nodes}
            for u in nodes:
                od = self.G.out_degree(u)
                if od == 0: continue
                sh = d * pr[u] / od
                for v in self.G.successors(u): new[v] += sh
            pr = new
        top = dict(sorted(pr.items(), key=lambda x: x[1], reverse=True)[:10])
        return {"algorithm": "PageRank", "top": top, "complexity": "O(k * E)"}

    def betweenness(self):
        nodes = list(self.G.nodes); CB = {v: 0.0 for v in nodes}
        for s in nodes:
            S, P = [], {v: [] for v in nodes}
            sigma = {v: 0 for v in nodes}; sigma[s] = 1
            d = {v: -1 for v in nodes}; d[s] = 0
            Q = deque([s])
            while Q:
                v = Q.popleft(); S.append(v)
                for w in self.G.successors(v):
                    if d[w] < 0:
                        Q.append(w); d[w] = d[v] + 1
                    if d[w] == d[v] + 1:
                        sigma[w] += sigma[v]; P[w].append(v)
            delta = {v: 0.0 for v in nodes}
            while S:
                w = S.pop()
                for v in P[w]:
                    delta[v] += (sigma[v]/sigma[w]) * (1 + delta[w])
                if w != s: CB[w] += delta[w]
        for v in CB: CB[v] /= 2.0
        top = dict(sorted(CB.items(), key=lambda x: x[1], reverse=True)[:10])
        return {"algorithm": "Brandes Betweenness", "top": top,
                "complexity": "O(V * E)"}

    def union_find_zones(self):
        parent = {n: n for n in self.G}; rank = {n: 0 for n in self.G}
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]; x = parent[x]
            return x
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra == rb: return
            if rank[ra] < rank[rb]: ra, rb = rb, ra
            parent[rb] = ra
            if rank[ra] == rank[rb]: rank[ra] += 1
        for u, v, d in self.G.edges(data=True):
            if d.get("edge_type") == "same_port": union(u, v)
        zones = defaultdict(list)
        for n in self.G: zones[find(n)].append(n)
        return {"algorithm": "Union-Find", "zone_count": len(zones),
                "complexity": "O(α(n)) per op"}

    def dp_knapsack(self, budget=10):
        nodes = list(self.G.nodes)
        costs = [1] * len(nodes)
        values = [self.G.nodes[n].get("fwd_packets", 1) for n in nodes]
        n = len(nodes)
        dp = [[0]*(budget+1) for _ in range(n+1)]
        for i in range(1, n+1):
            c, v = costs[i-1], values[i-1]
            for b in range(budget+1):
                dp[i][b] = dp[i-1][b]
                if c <= b:
                    dp[i][b] = max(dp[i][b], dp[i-1][b-c] + v)
        return {"algorithm": "DP Knapsack", "max_value": dp[n][budget],
                "budget": budget, "complexity": "O(n * W)"}

    def run_all(self, src=None, dst=None):
        if src is None and len(self.G) > 0: src = list(self.G.nodes)[0]
        if dst is None and len(self.G) > 1: dst = list(self.G.nodes)[-1]
        return {
            "graph_stats": {"nodes": self.G.number_of_nodes(),
                            "edges": self.G.number_of_edges(),
                            "density": nx.density(self.G) if len(self.G) > 1 else 0},
            "bfs": self.bfs(src),
            "dijkstra": self.dijkstra(src),
            "astar": self.astar(src, dst),
            "kahn_topo": self.kahn_topo(),
            "tarjan_scc": self.tarjan_scc(),
            "kruskal_mst": self.kruskal_mst(),
            "pagerank": self.pagerank(),
            "betweenness": self.betweenness(),
            "union_find": self.union_find_zones(),
            "dp_knapsack": self.dp_knapsack()
        }

In [10]:
def build_flow_graph(df, sample_size=300):
    """Build directed graph from real flow records — used both for training
    the anomaly detector AND for the graph-algorithm analysis of an incident."""
    sample = df.sample(n=min(sample_size, len(df)), random_state=42)
    G = nx.DiGraph()

    for idx, row in sample.iterrows():
        node_id = f"flow:{idx}"
        G.add_node(
            node_id,
            label=f"F{idx}",
            dest_port=int(row['Destination Port']),
            duration=float(row['Flow Duration']),
            fwd_packets=int(row['Total Fwd Packets']),
            attack_label=int(row['Label'] != 'BENIGN'),
            node_type="attack" if row['Label'] != 'BENIGN' else "benign",
        )

    nodes_by_port = defaultdict(list)
    for n, d in G.nodes(data=True):
        nodes_by_port[d['dest_port']].append(n)

    for port, nodes in nodes_by_port.items():
        for i in range(len(nodes) - 1):
            G.add_edge(nodes[i], nodes[i+1], edge_type="same_port",
                       port=port, duration=1.0, fwd_packets=1)
    return G

In [11]:
from google.colab import files
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print("Uploaded:", CSV_PATH)

Saving Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv to Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Uploaded: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [13]:
X_temp = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(df[feature_cols].mean())
X = X_temp.values

In [14]:
# ------------------------------------------------------------
# Robust cleaning: inf → NaN → median, then clip runaway values
# ------------------------------------------------------------
import numpy as np

X_df = df[feature_cols].copy()

# 1) Force numeric (any stray strings become NaN)
X_df = X_df.apply(pd.to_numeric, errors='coerce')

# 2) inf / -inf → NaN
X_df = X_df.replace([np.inf, -np.inf], np.nan)

# 3) Drop any column that is entirely NaN (avoids a NaN median)
X_df = X_df.dropna(axis=1, how='all')

# 4) Impute NaN with the MEDIAN (median is immune to inf-style outliers)
X_df = X_df.fillna(X_df.median(numeric_only=True))

# 5) Second-pass guard: if any inf survived, kill it now
X_df = X_df.replace([np.inf, -np.inf], np.nan)
X_df = X_df.fillna(0.0)

# 6) Convert and check for the "too large for float64" case
X = X_df.values.astype(np.float64)

# 7) Clip astronomical values to a sane percentile range
#    (CICIDS has flow-duration values up to ~1.2e8 and bytes/s up to 1e9;
#     anything beyond 1e15 is definitely garbage)
upper = np.percentile(X, 99.9, axis=0)
lower = np.percentile(X, 0.1, axis=0)
X = np.clip(X, lower, upper)

# 8) Final assertion — fail loudly if anything slipped through
assert np.isfinite(X).all(), "X still contains inf/NaN after cleaning!"
print(f"[clean] X shape={X.shape}, finite={np.isfinite(X).all()}")

[clean] X shape=(11287, 10), finite=True


In [15]:
y = (df['Label'] != 'BENIGN').astype(int).values

In [16]:
# ============================================================
# QUANTUM ANOMALY DETECTOR — FIXED DATA PIPELINE
# ============================================================
import os
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report

CSV_PATH = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"{CSV_PATH} not found in {os.getcwd()}")

df = pd.read_csv(CSV_PATH, low_memory=False)
df.columns = df.columns.str.strip()

print("=" * 70)
print("  QUANTUM ANOMALY DETECTOR (used to triage incident flow data)")
print("=" * 70)
print(f"Loaded rows: {len(df)}")

df = df.sample(frac=0.05, random_state=42).reset_index(drop=True)
print(f"Rows after 5% subsampling: {len(df)}")


# ------------------------------------------------------------
# Robust cleaning — this is the part that was broken
# ------------------------------------------------------------
feature_cols = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Fwd Packet Length Mean', 'Flow IAT Mean', 'Flow Packets/s',
    'Fwd Packets/s', 'Average Packet Size', 'Packet Length Variance',
    'SYN Flag Count'
]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

X_df = df[feature_cols].copy()
X_df = X_df.apply(pd.to_numeric, errors='coerce')          # strings -> NaN
X_df = X_df.replace([np.inf, -np.inf], np.nan)             # inf   -> NaN
X_df = X_df.dropna(axis=1, how='all')                      # drop all-NaN cols
X_df = X_df.fillna(X_df.median(numeric_only=True))         # NaN   -> median
X_df = X_df.replace([np.inf, -np.inf], np.nan).fillna(0.0) # belt & braces

X = X_df.values.astype(np.float64)

# Clip tails to kill the "too large for float64" offenders
upper = np.percentile(X, 99.9, axis=0)
lower = np.percentile(X, 0.1, axis=0)
X = np.clip(X, lower, upper)

assert np.isfinite(X).all(), "X still contains inf/NaN!"
print(f"[clean] X shape={X.shape}  finite={np.isfinite(X).all()}")

y = (df['Label'] != 'BENIGN').astype(int).values


# ------------------------------------------------------------
# Detector (unchanged)
# ------------------------------------------------------------
class QuantumAnomalyDetector:
    def __init__(self, n_qubits=4):
        self.n_qubits = n_qubits
        self.pca = PCA(n_components=n_qubits)
        self.scaler = StandardScaler()
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        Xs = self.scaler.fit_transform(X)
        self.X_train = self.pca.fit_transform(Xs)
        self.y_train = y
        return self

    def predict(self, X):
        Xp = self.pca.transform(self.scaler.transform(X))
        dists = cdist(Xp, self.X_train, metric='sqeuclidean')
        sims = np.exp(-dists)
        return self.y_train[np.argmax(sims, axis=1)]

    def anomaly_score(self, X):
        Xp = self.pca.transform(self.scaler.transform(X))
        benign = self.X_train[self.y_train == 0]
        if len(benign) == 0:
            return np.full(len(Xp), np.nan)
        return cdist(Xp, benign, metric='euclidean').min(axis=1)


# ------------------------------------------------------------
# Train / test
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

qdetector = QuantumAnomalyDetector(n_qubits=4).fit(X_train, y_train)
y_pred = qdetector.predict(X_test)
acc = (y_pred == y_test).mean()

print(f"\n[Quantum Detector] Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, target_names=['BENIGN', 'DDoS']))

  QUANTUM ANOMALY DETECTOR (used to triage incident flow data)
Loaded rows: 225745
Rows after 5% subsampling: 11287
[clean] X shape=(11287, 10)  finite=True

[Quantum Detector] Accuracy: 0.9970
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      1439
        DDoS       1.00      1.00      1.00      1948

    accuracy                           1.00      3387
   macro avg       1.00      1.00      1.00      3387
weighted avg       1.00      1.00      1.00      3387



In [17]:
class IncidentResponseOrchestrator:
    """
    End-to-end pipeline:
      1. Accept incident description + affected data types
      2. Select NIST SP 800-61 playbook
      3. Record every action in the tamper-evident timeline
      4. Run the graph algorithms on the flow graph (attack propagation)
      5. Evaluate regulatory notification triggers
      6. Produce compliance-ready report + timeline DataFrame
    """

    def __init__(self, description: str, data_types: List[str],
                 severity: str, incident_id: Optional[str] = None):
        self.incident_id = incident_id or f"IR-{uuid.uuid4().hex[:8].upper()}"
        self.description = description
        self.data_types = data_types
        self.severity = severity
        self.start_utc = datetime.now(timezone.utc)
        self.timeline = EvidenceTimeline(self.incident_id)
        self.regulations = RegulationEngine(self.start_utc)

        self.playbook_key, self.playbook = pick_playbook(description)

        # log the intake itself
        self.timeline.add_event(
            actor="intake-bot", action="incident_reported",
            details=description, nist_phase=NISTPhase.PREPARATION,
            attack_id="N/A")

    def run_playbook(self) -> List[Dict]:
        """Simulate each playbook step being executed and logged."""
        executed = []
        for step in self.playbook["steps"]:
            ev = self.timeline.add_event(
                actor="ir-runbook", action=step["action"],
                details=f"Playbook={self.playbook_key}",
                nist_phase=step["phase"], attack_id=step["attack"],
                artifact_hash=hashlib.sha256(step["action"].encode()).hexdigest()[:16]
            )
            executed.append(ev)
        return executed

    def analyze_graph(self, G: nx.DiGraph) -> Dict:
        """Graph-algorithm analysis of attack propagation over flow data."""
        engine = AttackGraphEngine(G)
        return engine.run_all()

    def regulatory_report(self) -> List[Dict]:
        return self.regulations.evaluate(self.data_types, self.severity)

    def phases_summary(self) -> pd.DataFrame:
        df = self.timeline.timeline_df()
        return (df.groupby("nist_phase")
                  .agg(steps=("seq","count"),
                       first_seen=("timestamp_utc","min"),
                       last_seen=("timestamp_utc","max"))
                  .reset_index())

    def attack_coverage(self) -> pd.DataFrame:
        df = self.timeline.timeline_df()
        cov = (df[df["attack_id"] != "N/A"])


In [18]:
# ============================================================
# LIVE 72-HOUR BREACH-NOTIFICATION CLOCK
# ============================================================
import time
import threading
from IPython.display import display, HTML, clear_output
from datetime import datetime, timezone, timedelta

class BreachClock:
    """
    Live countdown for GDPR Art. 33 (72h) + NIS2 (24h) + SEC (96h).
    Breach-notification clocks start at the moment of AWARENESS,
    not when the breach actually occurred.
    """

    def __init__(self, aware_utc: Optional[datetime] = None,
                 incident_id: str = "IR-DEMO"):
        self.aware_utc = aware_utc or datetime.now(timezone.utc)
        self.incident_id = incident_id
        self.deadlines = {
            "NIS2 (24h)":  self.aware_utc + timedelta(hours=24),
            "GDPR (72h)":  self.aware_utc + timedelta(hours=72),
            "SEC (96h)":   self.aware_utc + timedelta(hours=96),
        }

    @staticmethod
    def _fmt(td: timedelta) -> str:
        total = int(td.total_seconds())
        if total < 0:
            total = -total
            sign = "OVERDUE by "
        else:
            sign = ""
        d, rem = divmod(total, 86400)
        h, rem = divmod(rem, 3600)
        m, s  = divmod(rem, 60)
        return f"{sign}{d}d {h:02d}h {m:02d}m {s:02d}s"

    def _row_html(self, label: str, deadline: datetime) -> str:
        now = datetime.now(timezone.utc)
        remaining = deadline - now
        overdue = remaining.total_seconds() < 0
        # progress bar 0..100%
        total_span = (deadline - self.aware_utc).total_seconds()
        elapsed    = (now - self.aware_utc).total_seconds()
        pct = max(0.0, min(100.0, 100.0 * elapsed / total_span))
        color = "#e74c3c" if overdue else ("#f39c12" if pct > 75 else "#2ecc71")

        return f"""
        <div style="margin:8px 0; font-family:monospace;">
          <div style="display:flex; justify-content:space-between;">
            <b>{label}</b>
            <span style="color:{color}; font-weight:bold;">
              {self._fmt(remaining) if not overdue else self._fmt(remaining)}
            </span>
          </div>
          <div style="background:#333; height:10px; border-radius:5px; overflow:hidden; margin-top:4px;">
            <div style="background:{color}; width:{pct:.2f}%; height:100%;"></div>
          </div>
          <div style="font-size:11px; color:#888; margin-top:2px;">
            deadline: {deadline.isoformat()}  |  {pct:.2f}% elapsed
          </div>
        </div>
        """

    def render_html(self) -> str:
        rows = "".join(self._row_html(k, v) for k, v in self.deadlines.items())
        return f"""
        <div style="background:#111; color:#eee; padding:16px;
                    border-radius:10px; border:1px solid #444;
                    max-width:640px;">
          <h3 style="margin:0 0 4px 0;">⏱ Breach-Notification Clock</h3>
          <div style="font-size:12px; color:#888; margin-bottom:12px;">
            Incident <b>{self.incident_id}</b> —
            awareness at {self.aware_utc.isoformat()}
          </div>
          {rows}
        </div>
        """

    def run_live(self, refresh_seconds: float = 1.0,
                 duration_seconds: Optional[float] = None):
        """
        Refresh the widget every `refresh_seconds`.
        Pass duration_seconds=None to run indefinitely
        (press the Colab 'Stop' button to break).
        """
        t0 = time.time()
        try:
            while True:
                clear_output(wait=True)
                display(HTML(self.render_html()))
                if duration_seconds is not None and \
                   time.time() - t0 > duration_seconds:
                    break
                time.sleep(refresh_seconds)
        except KeyboardInterrupt:
            print("\n[clock stopped]")

In [19]:
class FastBreachClock(BreachClock):
    """
    Same as BreachClock but time is scaled: `speedup` simulated seconds
    pass for every 1 real second. Use for demos / demos in front of judges.
    """
    def __init__(self, aware_utc=None, incident_id="IR-DEMO", speedup=60.0):
        super().__init__(aware_utc, incident_id)
        self.speedup = speedup
        self._sim_start = datetime.now(timezone.utc)

    def _sim_now(self) -> datetime:
        real_elapsed = (datetime.now(timezone.utc) - self._sim_start).total_seconds()
        return self.aware_utc + timedelta(seconds=real_elapsed * self.speedup)

    def _row_html(self, label, deadline):
        now = self._sim_now()
        remaining = deadline - now
        overdue   = remaining.total_seconds() < 0
        total_span = (deadline - self.aware_utc).total_seconds()
        elapsed    = (now - self.aware_utc).total_seconds()
        pct = max(0.0, min(100.0, 100.0 * elapsed / total_span))
        color = "#e74c3c" if overdue else ("#f39c12" if pct > 75 else "#2ecc71")
        return f"""
        <div style="margin:8px 0; font-family:monospace;">
          <div style="display:flex; justify-content:space-between;">
            <b>{label}</b>
            <span style="color:{color}; font-weight:bold;">
              {self._fmt(remaining)}
            </span>
          </div>
          <div style="background:#333; height:10px; border-radius:5px; overflow:hidden; margin-top:4px;">
            <div style="background:{color}; width:{pct:.2f}%; height:100%;"></div>
          </div>
          <div style="font-size:11px; color:#888; margin-top:2px;">
            simulated now: {now.isoformat(timespec='seconds')} | {pct:.2f}% elapsed
          </div>
        </div>
        """


# --- DEMO: 1 real second = 60 simulated seconds, so 72h = 72 real minutes.
# To make it visible FAST for a demo, use speedup=3600 → 72h in ~72 real seconds.
clock = FastBreachClock(incident_id="IR-2026-0001", speedup=3600.0)
clock.run_live(refresh_seconds=1.0, duration_seconds=90)   # watch for 90s

In [ ]:
class OrchestratorClock:
    """Wraps BreachClock but pulls deadlines from the orchestrator's regulator engine."""

    def __init__(self, orch: "IncidentResponseOrchestrator", speedup: float = 60.0):
        self.orch = orch
        self.speedup = speedup
        self._sim_start = datetime.now(timezone.utc)
        # Build deadline map from the regulations actually triggered
        regs = orch.regulatory_report()
        self.deadlines = {}
        for r in regs:
            if r["deadline_hours"]:
                label = f"{r['regulation'].split('(')[0].strip()} ({r['deadline_hours']}h)"
                self.deadlines[label] = orch.start_utc + timedelta(hours=r["deadline_hours"])
        self.aware_utc = orch.start_utc
        self.incident_id = orch.incident_id

    def _sim_now(self):
        real_elapsed = (datetime.now(timezone.utc) - self._sim_start).total_seconds()
        return self.aware_utc + timedelta(seconds=real_elapsed * self.speedup)

    @staticmethod
    def _fmt(td: timedelta) -> str:
        total = int(td.total_seconds())
        sign = "OVERDUE by " if total < 0 else ""
        total = abs(total)
        d, rem = divmod(total, 86400)
        h, rem = divmod(rem, 3600)
        m, s  = divmod(rem, 60)
        return f"{sign}{d}d {h:02d}h {m:02d}m {s:02d}s"

    def render_html(self) -> str:
        now = self._sim_now()
        rows = ""
        for label, deadline in self.deadlines.items():
            remaining  = deadline - now
            overdue    = remaining.total_seconds() < 0
            total_span = (deadline - self.aware_utc).total_seconds()
            elapsed    = (now - self.aware_utc).total_seconds()
            pct = max(0.0, min(100.0, 100.0 * elapsed / total_span))
            color = "#e74c3c" if overdue else ("#f39c12" if pct > 75 else "#2ecc71")
            rows += f"""
            <div style="margin:8px 0; font-family:monospace;">
              <div style="display:flex; justify-content:space-between;">
                <b>{label}</b>
                <span style="color:{color}; font-weight:bold;">{self._fmt(remaining)}</span>
              </div>
              <div style="background:#333; height:10px; border-radius:5px; overflow:hidden; margin-top:4px;">
                <div style="background:{color}; width:{pct:.2f}%; height:100%;"></div>
              </div>
            </div>"""
        return f"""
        <div style="background:#111; color:#eee; padding:16px; border-radius:10px;
                    border:1px solid #444; max-width:640px;">
          <h3 style="margin:0 0 8px 0;">⏱ Live Notification Clock — {self.incident_id}</h3>
          <div style="font-size:12px; color:#888; margin-bottom:10px;">
            speedup = {self.speedup}× (1 real second = {self.speedup/60:.1f} sim minutes)
          </div>
          {rows}
        </div>"""

    def run_live(self, refresh_seconds=1.0, duration_seconds=None):
        t0 = time.time()
        try:
            while True:
                clear_output(wait=True)
                display(HTML(self.render_html()))
                if duration_seconds and time.time() - t0 > duration_seconds:
                    break
                time.sleep(refresh_seconds)
        except KeyboardInterrupt:
            print("\n[clock stopped]")


# --- Wire it up (assumes you already created an orchestrator) ---
orch = IncidentResponseOrchestrator(
    description="Ransomware encrypted file server; customer PII likely exfiltrated.",
    data_types=["personal_data", "pii", "customer_data"],
    severity="critical"
)

oclock = OrchestratorClock(orch, speedup=3600.0)  # 72h runs in ~72 real seconds
oclock.run_live(refresh_seconds=1.0, duration_seconds=90)